In [1]:
import pandas as pd 
import seaborn as sns 
from datetime import date


In [2]:
df = pd.read_csv("Airbnb_Open_Data.csv")

C:\Users\ks\AppData\Local\Temp\ipykernel_28372\3424017332.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Airbnb_Open_Data.csv")


In [6]:
df.isnull().sum()

id                                  0
NAME                              250
host id                             0
host_identity_verified            289
host name                         406
neighbourhood group                29
neighbourhood                      16
lat                                 8
long                                8
country                           532
country code                      131
instant_bookable                  105
cancellation_policy                76
room type                           0
Construction year                 214
price                             247
service fee                       273
minimum nights                    409
number of reviews                 183
review rate number                326
calculated host listings count    319
availability 365                  448
dtype: int64

In [8]:
# df = df.drop(columns={'license','house_rules','last review','reviews per month'})

In [9]:
df[['price', 'service fee']] = df[['price', 'service fee']].apply(pd.to_numeric, errors='coerce')


In [10]:
df['id'].duplicated().value_counts()

id
False    102058
True        541
Name: count, dtype: int64

In [11]:
df = df.drop_duplicates()

In [12]:
df.duplicated().sum()

np.int64(0)

In [14]:
df = df.dropna(subset=['Construction year'])

In [17]:
df['Construction year']

0         2020.0
1         2007.0
2         2005.0
3         2005.0
4         2009.0
           ...  
102047    2005.0
102048    2010.0
102049    2020.0
102050    2016.0
102057    2011.0
Name: Construction year, Length: 101844, dtype: float64

In [18]:
today = date.today()

In [19]:
df['Construction year'] = df['Construction year'].astype(int)


In [20]:
date_strings = df['Construction year'].astype(str) + '-01-01'


In [21]:
df['Construction Date'] = pd.to_datetime(date_strings)


In [22]:
today_timestamp = pd.to_datetime('today')


In [31]:
df['Age_Haus_Timedelta'] = today_timestamp - df['Construction Date']

In [32]:

print(df['Age_Haus_Timedelta']) # Immer noch timedelta64[ns]

0        2157 days 15:52:56.116711
1        6905 days 15:52:56.116711
2        7635 days 15:52:56.116711
3        7635 days 15:52:56.116711
4        6174 days 15:52:56.116711
                    ...           
102047   7635 days 15:52:56.116711
102048   5809 days 15:52:56.116711
102049   2157 days 15:52:56.116711
102050   3618 days 15:52:56.116711
102057   5444 days 15:52:56.116711
Name: Age_Haus_Timedelta, Length: 101844, dtype: timedelta64[ns]


In [33]:
df['Age_Haus_Timedelta']

0        2157 days 15:52:56.116711
1        6905 days 15:52:56.116711
2        7635 days 15:52:56.116711
3        7635 days 15:52:56.116711
4        6174 days 15:52:56.116711
                    ...           
102047   7635 days 15:52:56.116711
102048   5809 days 15:52:56.116711
102049   2157 days 15:52:56.116711
102050   3618 days 15:52:56.116711
102057   5444 days 15:52:56.116711
Name: Age_Haus_Timedelta, Length: 101844, dtype: timedelta64[ns]

In [34]:
df['Age_Haus_Timedelta'] = df['Age_Haus_Timedelta'].dt.days

In [35]:
df['Age_Haus_Timedelta']

0         2157
1         6905
2         7635
3         7635
4         6174
          ... 
102047    7635
102048    5809
102049    2157
102050    3618
102057    5444
Name: Age_Haus_Timedelta, Length: 101844, dtype: int64

In [43]:
# Welcher Bezirk ist am teuersten?
avg_price_by_neighbourhood = df.groupby('neighbourhood group')['price'].mean().sort_values(ascending=False)
print(avg_price_by_neighbourhood)


neighbourhood group
Bronx           NaN
Brooklyn        NaN
Manhattan       NaN
Queens          NaN
Staten Island   NaN
brookln         NaN
manhatan        NaN
Name: price, dtype: float64


In [37]:
# Welche Zimmerart wird am häufigsten angeboten?
count_by_room_type = df.groupby('room type')['id'].count()
print(count_by_room_type)


room type
Entire home/apt    53307
Hotel room           115
Private room       46215
Shared room         2207
Name: id, dtype: int64


In [38]:
# Haben Unterkünfte mit strenger Stornierung ein älteres Baujahr?
avg_age_by_policy = df.groupby('cancellation policy')['Construction year'].mean()
print(avg_age_by_policy)


KeyError: 'cancellation policy'

In [39]:
# Beispiel: Fehlende Preise durch den Median ersetzen
df['price'] = df['price'].fillna(df['price'].median())
df['service fee'] = df['service fee'].fillna(df['service fee'].median())


c:\Users\ks\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ks\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [41]:
# Wählen Sie die kategorialen Spalten aus, die Sie kodieren möchten
cols_to_encode = ['room type', 'neighbourhood group', 'cancellation_policy']

# Führen Sie One-Hot Encoding durch
df_encoded = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)
# drop_first=True hilft, Multikollinearität zu vermeiden


In [42]:
# Definieren Sie Ihre Zielvariable (Target)
y = df_encoded['price']

# Definieren Sie Ihre Features (Inputs) X
X = df_encoded[['minimum_nights', 'number_of_reviews', 'Age_Haus_Timedelta', 
                'room type_Private room', 'room type_Shared room', 
                'cancellation policy_moderate', 'cancellation policy_strict']]
# Fügen Sie hier alle relevanten kodierten Spalten hinzu


KeyError: "['minimum_nights', 'number_of_reviews', 'cancellation policy_moderate', 'cancellation policy_strict'] not in index"